# Phase 1e — Improving on Phase 1d (v2)

Phase 1d got S recall to **0.693** (the Random Forest managed 0.271) but left two
problems, and this notebook attacks both.

**Problem 1 — false alarms.** S precision was 0.294. Of DS2's 44,196 normal beats,
3,015 were called S — 98% of all the false positives.

**Problem 2 — we could not measure.** Fold-to-fold variance had std 0.13 while the
differences between arms were 0.007. Three of six arms could not be ranked at all.

### The four changes

| # | Change | Attacks |
| --- | --- | --- |
| 1 | Repeated CV over several seeds, **paired** comparison, bootstrap intervals | problem 2 |
| 2 | **Rhythm context** — a BiLSTM reads ±2 neighbouring beats | problem 1 |
| 3 | **Robust rhythm reference** — rolling median, not a record-level mean | problem 1 |
| 4 | **Ensembling** the fold models | problem 2 |

Change 2 is the substantive one. Phase 1d's model saw a single beat and had to
infer from its own RR numbers what rhythm it was early *against*. Now it can look.

### Arms

| arm | change |
| --- | --- |
| `beat_focal` | Phase 1d's winner, reproduced as the reference |
| `beat_robust_rr` | + robust rhythm reference (3) |
| `rhythm_context` | + neighbouring beats (2) |
| `rhythm_context_robust` | both (2 and 3) |

**This does not replace the Phase 1d notebook.** That one still runs as it did.

⏱️ **Expect roughly 2–3 hours.** The rhythm model is 3× larger and reads 5× more
data per sample. Lower `--seeds` to 2 or `--epochs` to 20 if you want it shorter.


## 1. Environment and GPU

`CUBLAS_WORKSPACE_CONFIG` is set before anything imports torch — CUDA reads it
when it builds its context, and setting it later does nothing.


In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['SMART_HEALTH_DATA'] = '/content/physionet-data'

!nvidia-smi -L || echo 'NO GPU -- Runtime -> Change runtime type -> GPU'

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Change the runtime type before continuing.')
print('device:', torch.cuda.get_device_name(0))


## 2. Upload the code

Rebuild the archive on your laptop first — it now contains the Phase 1e modules:

```
./venv/bin/python scripts/package_for_colab.py
```


In [ ]:
from google.colab import files
import zipfile, pathlib, shutil

uploaded = files.upload()
name = next(iter(uploaded))

work = pathlib.Path('/content/phase1e')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

with zipfile.ZipFile(name) as archive:
    archive.extractall(work)

%cd /content/phase1e
!ls src/ecg


## 3. Install dependencies


In [ ]:
!pip install -q -r requirements-colab.txt
import numpy, scipy, sklearn, wfdb
print('numpy', numpy.__version__, '| sklearn', sklearn.__version__,
      '| wfdb', wfdb.__version__)


## 4. GATE — tests must pass before any GPU time is spent

373 tests, about 10 seconds.


In [ ]:
import subprocess
if subprocess.run(['python', '-m', 'pytest', '-q'], text=True).returncode != 0:
    raise SystemExit('Tests failed. Do not train until this is green.')
print('\ntests green')


## 5a. Download MIT-BIH (retried)

PhysioNet drops connections often enough that this is retried automatically.
The script is idempotent — a re-run fetches only what is missing.


In [ ]:
import subprocess

for attempt in range(1, 6):
    print(f'--- download attempt {attempt} ---', flush=True)
    subprocess.run(['python', 'scripts/download_mitbih.py'], text=True)
    check = subprocess.run(['python', 'scripts/download_mitbih.py', '--verify'],
                           text=True, capture_output=True)
    if check.stdout:
        print(check.stdout.strip().splitlines()[-1])
    if check.returncode == 0:
        print('all records verify')
        break
    print('incomplete, retrying...', flush=True)
else:
    raise SystemExit('Download still incomplete after 5 attempts.')


## 5b. Preprocess


In [ ]:
!python scripts/preprocess_mitbih.py \
    --strategy ds1_ds2 --out-dir /content/physionet-data/processed/mitdb-ds12


## 5c. GATE — the data must match Phase 1d exactly

Same check as v1. A missing record shrinks DS2 and makes every number
incomparable to the results you already have.


In [ ]:
import json, pathlib, sys
sys.path.insert(0, 'src')
from ecg import config, dataset, splits

out = pathlib.Path('/content/physionet-data/processed/mitdb-ds12')
beats = out / 'beats'

n_npz = len(list(beats.glob('*.npz')))
assert n_npz == 48, f'expected 48 processed records, found {n_npz}'

assignment = json.loads((out / 'splits.json').read_text())
assert len(assignment['test']) == 22, (
    f"DS2 has {len(assignment['test'])} records, expected 22")

ds2 = dataset.load_records(beats, splits.DS2_RECORDS)
assert ds2.n_kept == 49646, f'DS2 has {ds2.n_kept} beats, expected 49,646'
print('DS2:', f'{ds2.n_kept:,}', 'beats -- matches Phase 1d exactly')


## 6. Sanity-check the rhythm-context windows

Two checks worth making before a two-hour run. The centre column must be the
beat itself, and **no window may contain beats from two patients** — that would
be a leakage bug presenting as a modelling result.


In [ ]:
import sys
sys.path.insert(0, 'src')
import numpy as np
from ecg import dataset, sequence, splits

beats_dir = '/content/physionet-data/processed/mitdb-ds12/beats'
sample = dataset.load_records(beats_dir, splits.DS1_RECORDS[:4])
seq = sequence.make_sequences(sample, context=2)

assert (seq.neighbours[:, 2] == np.arange(seq.n)).all(), 'centre column wrong'

crossings = sum(
    len(set(sample.record[seq.neighbours[i]])) > 1 for i in range(seq.n)
)
assert crossings == 0, f'{crossings} windows span two patients'

print(f'{seq.n:,} windows, width {seq.width}, no patient crossings')
print('window memory:', round(seq.neighbours.nbytes / 1e6, 1), 'MB',
      '(materialised would be',
      round(sample.X.nbytes * seq.width / 1e6), 'MB)')


## 7. Run the study

4 arms × 3 seeds × 5 folds = **60 training runs**, then final fits with
ensembling and one pass over DS2.

Progress is written to `nn_v2_cv.json` after every arm, so a dropped session
loses only the arm in flight.

Watch the `[paired]` block near the end — that is the analysis Phase 1d could not
do. `5/5 folds` means a consistent win; `mixed` means the change did nothing you
can distinguish from fold noise.


In [ ]:
!python scripts/train_nn_v2.py \
    --beats-dir /content/physionet-data/processed/mitdb-ds12/beats \
    --out-dir   /content/physionet-data/processed/mitdb-ds12 \
    --device cuda --epochs 30 --seeds 3 --folds 5 \
    2>&1 | tee /content/phase1e-run.log


### If you want it shorter

Stop the cell above and run this instead — 2 seeds and 20 epochs, roughly half
the time. The paired comparison still works, with less power.


In [ ]:
# !python scripts/train_nn_v2.py \
#     --beats-dir /content/physionet-data/processed/mitdb-ds12/beats \
#     --out-dir   /content/physionet-data/processed/mitdb-ds12 \
#     --device cuda --epochs 20 --seeds 2 --folds 5 \
#     2>&1 | tee /content/phase1e-run.log


## 8. Collect results and weights


In [ ]:
import shutil, pathlib, json

out = pathlib.Path('/content/physionet-data/processed/mitdb-ds12')
bundle = pathlib.Path('/content/phase1e-results')
if bundle.exists():
    shutil.rmtree(bundle)
(bundle / 'checkpoints_v2').mkdir(parents=True)

for name in ('nn_v2_cv.json', 'nn_v2_metrics.json', 'nn_folds.json'):
    src = out / name
    if src.exists():
        shutil.copy(src, bundle / name)
        print('collected', name)
    else:
        print('MISSING', name)

ckpt_dir = out / 'checkpoints_v2'
if ckpt_dir.exists():
    for c in sorted(ckpt_dir.glob('*.pt')):
        shutil.copy(c, bundle / 'checkpoints_v2' / c.name)
    print(f'collected {len(list(ckpt_dir.glob("*.pt")))} checkpoints')

shutil.copy('/content/phase1e-run.log', bundle / 'run.log')

shutil.make_archive('/content/phase1e-results', 'zip', bundle)
print('\nbundle:',
      round(pathlib.Path('/content/phase1e-results.zip').stat().st_size / 1e6, 2),
      'MB')


### Save to Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('/content/phase1e-results.zip',
            '/content/drive/MyDrive/phase1e-results.zip')
print('saved to Drive')


### Or download


In [ ]:
from google.colab import files
files.download('/content/phase1e-results.zip')


## 9. Read the headline before you leave

The one table that matters: did the rhythm context reduce false alarms?


In [ ]:
import json, pathlib

cv = json.loads(pathlib.Path(
    '/content/physionet-data/processed/mitdb-ds12/nn_v2_cv.json').read_text())

print(f"{'arm':<24}{'S F1':>8}{'S rec':>8}{'macroF1':>9}{'std':>7}")
print('-' * 56)
for name, row in sorted(cv['cv'].items(),
                        key=lambda kv: -kv[1]['mean_s_f1']):
    print(f"{name:<24}{row['mean_s_f1']:>8.3f}{row['mean_s_recall']:>8.3f}"
          f"{row['mean_macro_f1']:>9.3f}{row['std_macro_f1']:>7.3f}")

print('\nPaired vs beat_focal (consistent = won on every run):')
for arm, metrics in cv.get('paired_vs_reference', {}).items():
    s = metrics['s_f1']
    verdict = 'CONSISTENT' if s['consistent'] else 'mixed'
    print(f"  {arm:<24} S-F1 {s['wins']}/{s['n_folds']} "
          f"mean {s['mean_difference']:+.3f}  {verdict}")


## 10. Back on your laptop

```
./venv/bin/python scripts/ingest_colab_results.py ~/Downloads/phase1e-results.zip \
    --allow-arm-mismatch
```

`--allow-arm-mismatch` is needed because this bundle's arms are the Phase 1e
ones, not Phase 1d's — the ingest script checks that on purpose so a stale
bundle cannot quietly land in `results/`.
